<a href="https://colab.research.google.com/github/christophermagno/christophermagno/blob/main/Projects/Personal%20Health%20Analysis/health_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩻 Health Data Extraction

Using [**Garmin Connect API**](https://github.com/cyberjunky/python-garminconnect)

From Garmin watch

In [ ]:
# TODO: Gather solar data with timestamp ranges per day. Will need to store in a separate dataset and link through `Date`.

## ⬇️ Install Garmin python package

In [1]:
%pip install garminconnect

In [2]:
import re
import os
import sys
import logging
import importlib
import datetime
import requests
from getpass import getpass
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np

from google.colab import drive, userdata

from garth.exc import GarthException, GarthHTTPError
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)
log = logging.getLogger('Personal Health Analysis')

# Date format that Garmin likes
DATE_FORMAT = '%Y-%m-%d'

# Google Drive
drive.mount('/content/drive')

# Editable global vars
path_health_data = Path('/content/drive/MyDrive/Colab Notebooks/christophermagno/Projects/Personal Health Analysis/personal_health_data_2026.csv')
path_activities_data = Path('/content/drive/MyDrive/Colab Notebooks/christophermagno/Projects/Personal Health Analysis/personal_acitivites_data_2026.csv')

# Update the dataset even if the csv file exists
# Use caution because average heart rate data is gathered from granular
# minute-to-minute data which is only stored up to a certain date
# Only force updating the last 30 days to ensure we always have heart rate data
# and if we make updates to any of the previous days
FORCE_UPDATE = -30

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## ⌚️ Garmin helper functions to interact with API
Taken from example file provided in documentation to ensure safe usage with API

In [3]:
def safe_api_call(api_method, *args, **kwargs):
    """
    Safe API call wrapper with comprehensive error handling.

    This demonstrates the error handling patterns used throughout the library.
    Returns (success: bool, result: Any, error_message: str)
    """
    try:
        result = api_method(*args, **kwargs)
        return True, result, None

    except GarthHTTPError as e:
        # Handle specific HTTP errors gracefully
        error_str = str(e)
        status_code = getattr(getattr(e, "response", None), "status_code", None)

        if status_code == 400 or "400" in error_str:
            return (
                False,
                None,
                "Endpoint not available (400 Bad Request) - Feature may not be enabled for your account",
            )
        elif status_code == 401 or "401" in error_str:
            return (
                False,
                None,
                "Authentication required (401 Unauthorized) - Please re-authenticate",
            )
        elif status_code == 403 or "403" in error_str:
            return (
                False,
                None,
                "Access denied (403 Forbidden) - Account may not have permission",
            )
        elif status_code == 404 or "404" in error_str:
            return (
                False,
                None,
                "Endpoint not found (404) - Feature may have been moved or removed",
            )
        elif status_code == 429 or "429" in error_str:
            return (
                False,
                None,
                "Rate limit exceeded (429) - Please wait before making more requests",
            )
        elif status_code == 500 or "500" in error_str:
            return (
                False,
                None,
                "Server error (500) - Garmin's servers are experiencing issues",
            )
        elif status_code == 503 or "503" in error_str:
            return (
                False,
                None,
                "Service unavailable (503) - Garmin's servers are temporarily unavailable",
            )
        else:
            return False, None, f"HTTP error: {e}"

    except FileNotFoundError:
        return (
            False,
            None,
            "No valid tokens found. Please login with your email/password to create new tokens.",
        )

    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication issue: {e}"

    except GarminConnectConnectionError as e:
        return False, None, f"Connection issue: {e}"

    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit exceeded: {e}"

    except Exception as e:
        return False, None, f"Unexpected error: {e}"

def get_credentials():
    """Get email and password from environment or user input."""
    email = os.getenv("EMAIL") or userdata.get('GARMIN_EMAIL')
    password = os.getenv("PASSWORD") or userdata.get('GARMIN_PASS')

    if not email:
        email = input("Login email: ")
    if not password:
        password = getpass("Enter password: ")

    return email, password

def init_api() -> Garmin | None:
    """Initialize Garmin API with authentication and token management."""

    # Configure token storage
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = Path(tokenstore).expanduser()

    print(f"🔐 Token storage: {tokenstore_path}")

    # Check if token files exist
    if tokenstore_path.exists():
        print("📄 Found existing token directory")
        token_files = list(tokenstore_path.glob("*.json"))
        if token_files:
            print(
                f"🔑 Found {len(token_files)} token file(s): {[f.name for f in token_files]}"
            )
        else:
            print("⚠️ Token directory exists but no token files found")
    else:
        print("📭 No existing token directory found")

    # First try to login with stored tokens
    try:
        print("🔄 Attempting to use saved authentication tokens...")
        garmin = Garmin()
        garmin.login(str(tokenstore_path))
        print("✅ Successfully logged in using saved tokens!")
        return garmin

    except (
        FileNotFoundError,
        GarthHTTPError,
        GarminConnectAuthenticationError,
        GarminConnectConnectionError,
    ):
        print("🔑 No valid tokens found. Requesting fresh login credentials.")

    # Loop for credential entry with retry on auth failure
    while True:
        try:
            # Get credentials
            email, password = get_credentials()

            print("� Logging in with credentials...")
            garmin = Garmin(
                email=email, password=password, is_cn=False, return_on_mfa=True
            )
            result1, result2 = garmin.login()

            if result1 == "needs_mfa":
                print("🔐 Multi-factor authentication required")

                mfa_code = input("Please enter your MFA code: ")
                print("🔄 Submitting MFA code...")

                try:
                    garmin.resume_login(result2, mfa_code)
                    print("✅ MFA authentication successful!")

                except GarthHTTPError as garth_error:
                    # Handle specific HTTP errors from MFA
                    error_str = str(garth_error)
                    if "429" in error_str and "Too Many Requests" in error_str:
                        print("❌ Too many MFA attempts")
                        print("💡 Please wait 30 minutes before trying again")
                        sys.exit(1)
                    elif "401" in error_str or "403" in error_str:
                        print("❌ Invalid MFA code")
                        print("💡 Please verify your MFA code and try again")
                        continue
                    else:
                        # Other HTTP errors - don't retry
                        print(f"❌ MFA authentication failed: {garth_error}")
                        sys.exit(1)

                except GarthException as garth_error:
                    print(f"❌ MFA authentication failed: {garth_error}")
                    print("💡 Please verify your MFA code and try again")
                    continue

            # Save tokens for future use
            garmin.garth.dump(str(tokenstore_path))
            print(f"💾 Authentication tokens saved to: {tokenstore_path}")
            print("✅ Login successful!")
            return garmin

        except GarminConnectAuthenticationError:
            print("❌ Authentication failed:")
            print("💡 Please check your username and password and try again")
            # Continue the loop to retry
            continue

        except (
            FileNotFoundError,
            GarthHTTPError,
            GarminConnectConnectionError,
            requests.exceptions.HTTPError,
        ) as err:
            print(f"❌ Connection error: {err}")
            print("💡 Please check your internet connection and try again")
            return None

        except KeyboardInterrupt:
            print("\n👋 Cancelled by user")
            return None


### Get Garmin client

In [4]:
api = init_api()

🔐 Token storage: /root/.garminconnect
📄 Found existing token directory
🔑 Found 2 token file(s): ['oauth2_token.json', 'oauth1_token.json']
🔄 Attempting to use saved authentication tokens...
✅ Successfully logged in using saved tokens!


### A Quick Test
Will fail even though loggined in successfully but not initialized properly

In [5]:
api.get_stats('2026-01-01')

{'userProfileId': 126748136,
 'totalKilocalories': 3123.0,
 'activeKilocalories': 1612.0,
 'bmrKilocalories': 1511.0,
 'wellnessKilocalories': 3123.0,
 'burnedKilocalories': None,
 'consumedKilocalories': None,
 'remainingKilocalories': None,
 'totalSteps': 10564,
 'netCalorieGoal': None,
 'totalDistanceMeters': 7586,
 'wellnessDistanceMeters': 7586,
 'wellnessActiveKilocalories': 1612.0,
 'netRemainingKilocalories': 1612.0,
 'userDailySummaryId': 126748136,
 'calendarDate': '2026-01-01',
 'rule': {'typeId': 2, 'typeKey': 'private'},
 'uuid': '9c0ea781368e44a48875fb994e4a861e',
 'dailyStepGoal': 8230,
 'wellnessStartTimeGmt': '2026-01-01T08:00:00.0',
 'wellnessStartTimeLocal': '2026-01-01T00:00:00.0',
 'wellnessEndTimeGmt': '2026-01-02T08:00:00.0',
 'wellnessEndTimeLocal': '2026-01-02T00:00:00.0',
 'durationInMilliseconds': 86400000,
 'wellnessDescription': None,
 'highlyActiveSeconds': 2036,
 'activeSeconds': 16485,
 'sedentarySeconds': 34519,
 'sleepingSeconds': 33360,
 'includesWell

## Helper functions for datetime

In [6]:
def convert_epoch_to_datetime(epoch):
    """
    Convert epoch miliseconds to datetime.
    """
    try:
        return datetime.datetime.fromtimestamp(epoch / 1000)
    except TypeError:
        return pd.NaT

def get_date_range(start=None, rng=None):
    """
    Get a range of dates in garmin friendly date string.
    :param start: datetime.date
    :return: list of dates in garmin friendly date string
    """
    start = start or datetime.datetime.today()
    rng = rng or int(start.strftime('%j'))
    dates = reversed([start - datetime.timedelta(days=x) for x in range(rng)])
    return [x.strftime(DATE_FORMAT) for x in dates]


## Helper functions to gather and organize Garmin data

Some useful methods from the Garmin class to use
* get_stats - using
* get_heart_rates - using
* get_sleep_data - using
* get_fitnessage_data - using
* get_activities - using
*
* get_steps_data
* get_daily_steps
* get_floors
* get_stress_data
* get_rhr_day
* get_hrv_data

Others to look at
* get_activities_fordate
* get_earned_badges

In [7]:
def get_device():
    """
    deviceId
    imageUrl
    productDisplayName
    deviceTypeSimpleName
    """
    devices = safe_api_call(api.get_devices)[1]
    return devices[0]

def get_sleep_data(date):

    sleep_data = {}

    to_pop = [
        'id',
        'userProfilePK',
        'napTimeSeconds',
        'sleepWindowConfirmed',
        'sleepWindowConfirmationType',
        'autoSleepStartTimestampGMT',
        'autoSleepEndTimestampGMT',
        'sleepQualityTypePK',
        'sleepResultTypePK',
        'deviceRemCapable',
        'retro',
        'sleepFromDevice',
        'sleepScores',
        'sleepScoreInsight',
        'sleepScorePersonalizedInsight',
        'sleepVersion',
        'averageSPO2',
        'averageSpO2HR',
        'lowestSPO2'
    ]

    data = safe_api_call(api.get_sleep_data, date)[1]

    sleep_data.update(data['dailySleepDTO'])
    if 'sleepScores' in sleep_data:
        sleep_data['sleepScore'] = sleep_data['sleepScores']['overall']['value']
        sleep_data['sleepScoreQuality'] = sleep_data['sleepScores']['overall']['qualifierKey']
        sleep_data['stressSleepQuality'] = sleep_data['sleepScores']['stress']['qualifierKey']
        sleep_data['awakeCountQuality'] = sleep_data['sleepScores']['awakeCount']['qualifierKey']
        sleep_data['remSleepQuality'] = sleep_data['sleepScores']['remPercentage']['qualifierKey']
        sleep_data['restlessnessSleepQuality'] = sleep_data['sleepScores']['restlessness']['qualifierKey']
        sleep_data['lightSleepQuality'] = sleep_data['sleepScores']['lightPercentage']['qualifierKey']
        sleep_data['deepSleepQuality'] = sleep_data['sleepScores']['deepPercentage']['qualifierKey']
    else:
        sleep_data['sleepScore'] = None
        sleep_data['sleepScoreQuality'] = None
        sleep_data['stressSleepQuality'] = None
        sleep_data['awakeCountQuality'] = None
        sleep_data['remSleepQuality'] = None
        sleep_data['restlessnessSleepQuality'] = None
        sleep_data['lightSleepQuality'] = None
        sleep_data['deepSleepQuality'] = None

    # result['sleepHeartRate'] = data['sleepHeartRate']
    sleep_data['avgOvernightHrv'] = data.get('avgOvernightHrv')
    sleep_data['hrvStatus'] = data.get('hrvStatus')
    sleep_data['restingHeartRate'] = data.get('restingHeartRate')

    # Convert timestamp to datetime
    for item in ['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'sleepStartTimestampLocal', 'sleepEndTimestampLocal']:
        sleep_data[item] = convert_epoch_to_datetime(sleep_data[item])

    for key in to_pop:
        try:
            sleep_data.pop(key)
        except KeyError as e:
            pass

    return sleep_data

def get_hydration_data(date):
    data = safe_api_call(api.get_hydration_data, date)[1]
    hydration_data = {
        'hydrationValueInML': data['valueInML'],
        'hydrationGoalInML': data['goalInML'],
        'sweatLossInML': data['sweatLossInML']
    }
    return hydration_data

def get_solar_data():
    """
    Example output
    {
    'localConnectDate': '2025-12-28',
     'userProfilePk': 126748136,
     'deviceId': 3476417250,
     'solarInputReadings': [{'readingTimestampLocal': '2025-12-28T00:00:00.0',
       'readingTimestampGmt': '2025-12-28T08:00:00.0',
       'solarUtilization': 0.0,
       'notChargingTooHot': False,
       'notChargingTooCold': False,
       'notChargingBatteryFull': False,
       'notChargingExternalPower': False,
       'notChargingUserDisabled': False,
       'notChargingOther': True,
       'activityTimeGainMs': 0,
       'charging': False,
       'interpolated': None},
    """

    d = safe_api_call(api.get_device_solar_data, get_device()['deviceId'], dates[-3], dates[-1])[1]['solarDailyDataDTOs']
    print(d)

### Get Activities Data

In [8]:
def get_activities_data(dates):
    """
    - ownerFullName
    - ownerProfileImageUrlMedium

    activityId
    activityName
    activityType: {
        typeId
        typeKey
    }
    locationName (hiking?)
    startTimeLocal
    startTimeGMT
    endTimeGMT
    beginTimestamp

    distance
    duration
    elapsedDuration
    movingDuration
    - elevationGain (hiking)
    - elevationLoss (hiking)
    averageSpeed
    maxSpeed
    hasPolyline
    hasImages
    ownerId
    ownerFullName
    ownerProfileImageUrlMedium
    calories
    bmrCalories
    averageHR
    maxHR
    steps
    aerobicTrainingEffect
    anaerobicTrainingEffect
    summarizedExerciseSets: [
        category
        reps
        volume
        duration
        sets
        maxWeight
    ]

    lapCount
    totalSets
    activeSets
    totalReps

    activityTrainingLoad
    minActivityLapDuration

    moderateIntensityMinutes
    vigorousIntensityMinutes

    hrTimeInZone_1
    hrTimeInZone_2
    hrTimeInZone_3
    hrTimeInZone_4
    hrTimeInZone_5

    pr
    """

    result = []

    keys_to_get = [
        'activityId',
        'activityName',
        # activityType: {
        #     typeId
        #     typeKey
        # }

        # 'ownerId',
        # 'ownerFullName',
        # 'ownerProfileImageUrlMedium',
        # 'hasImages',

        # 'locationName',  # (hiking?)
        # 'elevationGain',  # (hiking)
        # 'elevationLoss',  # (hiking)

        'startTimeLocal',
        'startTimeGMT',
        'endTimeGMT',
        'beginTimestamp',

        'pr',

        'duration',
        'elapsedDuration',
        'movingDuration',

        'calories',
        'bmrCalories',

        'steps',
        'distance',

        'averageSpeed',
        'maxSpeed',

        'averageHR',
        'maxHR',
        'hrTimeInZone_1',
        'hrTimeInZone_2',
        'hrTimeInZone_3',
        'hrTimeInZone_4',
        'hrTimeInZone_5',

        'lapCount',
        'totalSets',
        'activeSets',
        'totalReps',

        'aerobicTrainingEffect',
        'anaerobicTrainingEffect',
        'moderateIntensityMinutes',
        'vigorousIntensityMinutes',
        'activityTrainingLoad'
    ]
    activities = safe_api_call(api.get_activities_by_date, dates[0], dates[-1])[1]
    for activity in activities:
        activity_data = {}
        for key in keys_to_get:
            activity_data[key] = activity.get(key, None)
            if key == 'activityName':
                activity_data['activityType'] = activity['activityType'].get('typeKey', 'Unknown').title().replace('_', ' ')

            # Get total calories
            if key == 'bmrCalories':
                activity_data['totalCalories'] = activity['bmrCalories'] + activity['calories']

        result.append(activity_data)
    return result

### Get Health Data

In [9]:
def get_health_data(date):
    """
    """

    to_pop = [
        'userProfileId',
        'userDailySummaryId',
        'burnedKilocalories',
        'wellnessActiveKilocalories',
        'netRemainingKilocalories',
        'rule',
        'wellnessStartTimeGmt',
        'wellnessStartTimeLocal',
        'wellnessEndTimeGmt',
        'wellnessEndTimeLocal',
        'durationInMilliseconds',
        'wellnessDescription',
        'includesWellnessData',
        'includesActivityData',
        'includesCalorieConsumedData',
        'privacyProtected',
        'floorsAscended',
        'floorsDescended',
        'lastSevenDaysAvgRestingHeartRate',
        'source',
        'lastSyncTimestampGMT',
        'bodyBatteryMostRecentValue',
        'bodyBatteryVersion',
        # 'averageSpo2',
        'lowestSpO2Value',
        'highestSpO2Value',
        'lowestSpo2',
        'latestSpo2',
        'latestSpo2ReadingTimeGmt',
        'latestSpo2ReadingTimeLocal',
        'latestSpo2ReadingTimeLocalaverageMonitoringEnvironmentAltitude',
        'restingCaloriesFromActivity',
        'latestRespirationValue',
        'latestRespirationTimeGMT',
        'respirationAlgorithmVersion',
        'ageGroup',
        'averageMonitoringEnvironmentAltitude',
        # 'bodyBatteryChargedValue',
        # 'bodyBatteryDrainedValue',
        # 'bodyBatteryHighestValue',
        # 'bodyBatteryLowestValue',
        # 'bodyBatteryDuringSleep',
        'wellnessKilocalories',
        'consumedKilocalories',
        'remainingKilocalories',
        'netCalorieGoal',
        'wellnessDistanceMeters',
        'userNote',
        'sleepingSeconds',
        'minAvgHeartRate',
        'maxAvgHeartRate',
        'abnormalHeartRateAlertsCount',
        'unmeasurableSleepSeconds',
        # 'measurableAsleepDuration',
        # 'measurableAwakeDuration',
        'stressPercentage',
        'restStressPercentage',
        'activityStressPercentage',
        'uncategorizedStressPercentage',
        'lowStressPercentage',
        'mediumStressPercentage',
        'highStressPercentage',
        'restStressDuration',
        'userFloorsAscendedGoal',
    ]

    success, health_data, _ = safe_api_call(api.get_stats, date)
    success, result, _ = safe_api_call(api.get_fitnessage_data, date)
    if success:
        health_data['fitnessAge'] = int(result['fitnessAge'])
    else:
        health_data['fitnessAge'] = None

    # Get heart rate data

    # Heart rate data
    hdata = safe_api_call(api.get_heart_rates, date)[1]['heartRateValues']
    if hdata:
        heart_rates = [v[1] for v in hdata if v[1] is not None]
        health_data['avgHeartRate'] = float(np.array(heart_rates).mean())

    # Get sleep data
    health_data.update(get_sleep_data(date))

    # Get hydration data
    health_data.update(get_hydration_data(date))

    for key in to_pop:
        try:
            health_data.pop(key)
        except KeyError as e:
            pass

    # Convert/Add some columns
    convert_dict = {}
    for key, value in health_data.items():
        if value:
            if 'Meters' in key:
                convert_dict[key.replace('Meters', 'Miles')] = value / 1609
            elif 'Seconds' in key:
                convert_dict[key.replace('Seconds', 'Hours')] = value / 3600
            elif 'Duration' in key:
                convert_dict[key.replace('Duration', 'Hours')] = value / 3600
            elif 'Minutes' in key:
                convert_dict[key.replace('Minutes', 'Hours')] = value / 60
            elif 'InML' in key:
                convert_dict[key.replace('InML', 'InCups')] = value / 240

    # Update
    health_data.update(convert_dict)

    return health_data

In [10]:
def get_healths_data(dates=None):
    data = []
    for date in tqdm(dates or get_date_range()):
        data.append(get_health_data(date))
    return data

## 📆 Get Date Range

In [11]:
# current_date = datetime.datetime.strptime('12-31-2025', '%m-%d-%Y')
current_date = datetime.datetime.today()
dates = get_date_range(current_date)
log.info(f'Number of dates: {len(dates)}')

2026-01-22 18:41:12,262 - INFO - Number of dates: 22


## ⚕️ Build Health Data
### Create the Health Dataframe

Remapping dictionary to fix the columns names and order

In [12]:
remap_health_columns = {
    'uuid': 'uuid',
    'calendarDate': 'Date',

    'fitnessAge': 'Fitness Age',

    # Calories
    'totalKilocalories': 'Calories',
    'activeKilocalories': 'Active Calories',
    'bmrKilocalories': 'Resting Calories',

    # Hydration
    'hydrationValueInML': 'Hydration Value In ML',
    'hydrationGoalInML': 'Hydration Goal In ML',
    'sweatLossInML': 'Sweat Loss In ML',
    'hydrationValueInCups': 'Hydration Value In Cups',
    'hydrationGoalInCups': 'Hydration Goal In Cups',
    'sweatLossInCups': 'Sweat Loss In Cups',

    # Heart Rate
    'avgHeartRate': 'Average Heart Rate',
    'minHeartRate': 'Min Heart Rate',
    'maxHeartRate': 'Max Heart Rate',
    'restingHeartRate': 'Resting Heart Rate',
    'hrvStatus': 'Heart Rate Variability Qualifier',

    # Peripheral Oxyge Saturation
    'averageSpo2': 'Average Sp 02',

    # Respiration
    'avgWakingRespirationValue': 'Avg Waking Respiration Value',
    'highestRespirationValue': 'Highest Respiration Value',
    'lowestRespirationValue': 'Lowest Respiration Value',

    # Steps/Distance
    'totalSteps': 'Total Steps',
    'totalDistanceMeters': 'Total Distance Meters',
    'totalDistanceMiles': 'Total Distance Miles',
    'dailyStepGoal': 'Daily Step Goal',

    # Floors
    'floorsAscendedInMeters': 'Floors Ascended In Meters',
    'floorsDescendedInMeters': 'Floors Descended In Meters',

    'floorsAscendedInMiles': 'Floors Ascended In Miles',
    'floorsDescendedInMiles': 'Floors Descended In Miles',

    # Activity
    'activeSeconds': 'Active Seconds',
    'highlyActiveSeconds': 'Highly Active Seconds',
    'sedentarySeconds': 'Sedentary Seconds',
    'moderateIntensityMinutes': 'Moderate Intensity Minutes',
    'vigorousIntensityMinutes': 'Vigorous Intensity Minutes',
    'intensityMinutesGoal': 'Intensity Minutes Goal',

    'activeHours': 'Active Hours',
    'highlyActiveHours': 'Highly Active Hours',
    'sedentaryHours': 'Sedentary Hours',
    'moderateIntensityHours': 'Moderate Intensity Hours',
    'vigorousIntensityHours': 'Vigorous Intensity Hours',
    'intensityHoursGoal': 'Intensity Hours Goal',

    # Stress
    'averageStressLevel': 'Average Stress Level',
    'totalStressDuration': 'Total Stress Seconds',
    'stressDuration': 'Stress Seconds',
    'maxStressLevel': 'Max Stress Level',
    'uncategorizedStressDuration': 'Uncategorized Stress Seconds',
    'lowStressDuration': 'Low Stress Seconds',
    'mediumStressDuration': 'Medium Stress Seconds',
    'highStressDuration': 'High Stress Seconds',
    'activityStressDuration': 'Activity Stress Seconds',
    'stressQualifier': 'Stress Qualifier',

    'stressHours': 'Stress Hours',
    'activityStressHours': 'Activity Stress Hours',
    'uncategorizedStressHours': 'Uncategorized Stress Hours',
    'totalStressHours': 'Total Stress Hours',
    'lowStressHours': 'Low Stress Hours',
    'mediumStressHours': 'Medium Stress Hours',
    'highStressHours': 'High Stress Hours',

    # Body battery
    'bodyBatteryAtWakeTime': 'Body Battery',
    'bodyBatteryChargedValue': 'Body Battery Charged Value',
    'bodyBatteryDrainedValue': 'Body Battery Drained Value',
    'bodyBatteryHighestValue': 'Body Battery Highest Value',
    'bodyBatteryLowestValue': 'Body Battery Lowest Value',
    'bodyBatteryDuringSleep': 'Body Battery During Sleep',

    # Sleep data
    'sleepStartTimestampGMT': 'Sleep Start Timestamp GMT',
    'sleepEndTimestampGMT': 'Sleep End Timestamp GMT',
    'sleepStartTimestampLocal': 'Sleep Start Timestamp Local',
    'sleepEndTimestampLocal': 'Sleep End Timestamp Local',

    'sleepTimeSeconds': 'Sleep Time Seconds',
    'sleepScore': 'Sleep Score',
    'sleepScoreQuality': 'Sleep Quality',
    'sleepScoreFeedback': 'Sleep Feedback',

    'measurableAsleepDuration': 'Measurable Asleep Seconds',
    'measurableAwakeDuration': 'Measurable Awake Seconds',

    'lightSleepSeconds': 'Light Sleep Seconds',
    'deepSleepSeconds': 'Deep Sleep Seconds',
    'remSleepSeconds': 'Rem Sleep Seconds',
    'awakeSleepSeconds': 'Awake Sleep Seconds',

    'sleepTimeHours': 'Sleep Time Hours',
    'measurableAsleepHours': 'Measurable Asleep Hours',
    'measurableAwakeHours': 'Measurable Awake Hours',
    'deepSleepHours': 'Deep Sleep Hours',
    'lightSleepHours': 'Light Sleep Hours',
    'remSleepHours': 'Rem Sleep Hours',
    'awakeSleepHours': 'Awake Sleep Hours',

    'averageRespirationValue': 'Average Respiration Value',
    'awakeCount': 'Awake Count',
    'avgSleepStress': 'Avg Sleep Stress',

    'stressSleepQuality': 'Stress Sleep Quality',
    'awakeCountQuality': 'Awake Count Quality',
    'remSleepQuality': 'Rem Sleep Quality',
    'restlessnessSleepQuality': 'Restlessness Sleep Quality',
    'lightSleepQuality': 'Light Sleep Quality',
    'deepSleepQuality': 'Deep Sleep Quality',

    'avgOvernightHrv': 'Average Overnight Hrv',
}

### 🧱 Generate the Health Data

In [16]:
# Build the health data set
# If data already exists, just read the data and generate health data for
# new days
if path_health_data.exists():
    df_health = pd.read_csv(path_health_data)
    try:
        df_health = df_health.drop('Unnamed: 0', axis=1)  # I don't know why it keeps adding this column...
    except:
        pass

    # dates_to_update = sorted(list(set(dates).difference(set(df_health['Date'].tolist()))))

    # Only update the previous day(s) and not current day since we're still
    # gathering data for that day
    dates_to_update = dates[FORCE_UPDATE:-1]

    log.info(f'Updating Dates: {dates_to_update}')
    health_data = get_healths_data(dates_to_update)

    # Rename the dictionary keys
    for item in health_data:
        for key, value in remap_health_columns.items():
            try:
                item[value] = item.pop(key)
            except KeyError:
                pass

    # Concatenate the existing and new DataFrames
    df_health = pd.concat([df_health.iloc[:FORCE_UPDATE], pd.DataFrame(health_data).convert_dtypes()]).reset_index()
else:
    # Otherwise generate the data YTD
    health_data = get_healths_data(dates)
    df_health = pd.DataFrame(health_data).convert_dtypes()
df_health.tail()

2026-01-22 18:43:11,901 - INFO - Updating Dates: ['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04', '2026-01-05', '2026-01-06', '2026-01-07', '2026-01-08', '2026-01-09', '2026-01-10', '2026-01-11', '2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15', '2026-01-16', '2026-01-17', '2026-01-18', '2026-01-19', '2026-01-20', '2026-01-21']
100%|██████████| 21/21 [00:30<00:00,  1.46s/it]
/tmp/ipython-input-2911529505.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_health = pd.concat([df_health.iloc[:FORCE_UPDATE], pd.DataFrame(health_data).convert_dtypes()]).reset_index()


,index,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,...,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv,averageSpO2Value,averageSpO2HRSleep
16,16,b00f2774b1ca44e0aeb1e0dcf3096270,2026-01-17,32,1799,288,1511,1656.116,2865.056,26,...,17,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,EXCELLENT,EXCELLENT,44,<NA>,<NA>
17,17,8ee13862ae9b4afab23fe95f4905f627,2026-01-18,32,2916,1405,1511,2839.056,2839.056,<NA>,...,26,POOR,GOOD,GOOD,EXCELLENT,FAIR,POOR,44,<NA>,<NA>
18,18,980217c4d706469d907be131f329cb0b,2026-01-19,32,2472,961,1511,2839.056,2839.056,<NA>,...,21,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,EXCELLENT,GOOD,48,<NA>,<NA>
19,19,7b57bcd8614a4bbcbef6bc188c82f0a4,2026-01-20,32,1866,355,1511,1892.704,2839.056,<NA>,...,22,FAIR,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,55,<NA>,<NA>
20,20,ad9c64b6ad374864adb28ecc06683bf2,2026-01-21,32,1701,190,1511,1656.116,2839.056,<NA>,...,23,FAIR,GOOD,EXCELLENT,EXCELLENT,GOOD,FAIR,43,<NA>,<NA>


### Rename the columns

In [17]:
df_health1 = df_health.rename(remap_health_columns, axis=1)[remap_health_columns.values()]

In [18]:
df_health1.tail()

,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,Hydration Value In Cups,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
16,b00f2774b1ca44e0aeb1e0dcf3096270,2026-01-17,32,1799,288,1511,1656.116,2865.056,26,6.900483,...,14,0,17,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,EXCELLENT,EXCELLENT,44
17,8ee13862ae9b4afab23fe95f4905f627,2026-01-18,32,2916,1405,1511,2839.056,2839.056,<NA>,11.8294,...,14,1,26,POOR,GOOD,GOOD,EXCELLENT,FAIR,POOR,44
18,980217c4d706469d907be131f329cb0b,2026-01-19,32,2472,961,1511,2839.056,2839.056,<NA>,11.8294,...,14,0,21,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,EXCELLENT,GOOD,48
19,7b57bcd8614a4bbcbef6bc188c82f0a4,2026-01-20,32,1866,355,1511,1892.704,2839.056,<NA>,7.886267,...,13,0,22,FAIR,EXCELLENT,FAIR,EXCELLENT,EXCELLENT,EXCELLENT,55
20,ad9c64b6ad374864adb28ecc06683bf2,2026-01-21,32,1701,190,1511,1656.116,2839.056,<NA>,6.900483,...,15,1,23,FAIR,GOOD,EXCELLENT,EXCELLENT,GOOD,FAIR,43


### Prettify the qualifer/feedback text columns

In [19]:
title_columns = ['Qualifier', 'Quality', 'Feedback']
for col, series in df_health1.items():
    for item in title_columns:
        if re.search(item, col, re.IGNORECASE):
            df_health1[col] = series.str.title().str.replace('_', ' ')
df_health1.tail()

,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,Hydration Value In Cups,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
16,b00f2774b1ca44e0aeb1e0dcf3096270,2026-01-17,32,1799,288,1511,1656.116,2865.056,26,6.900483,...,14,0,17,Fair,Excellent,Excellent,Excellent,Excellent,Excellent,44
17,8ee13862ae9b4afab23fe95f4905f627,2026-01-18,32,2916,1405,1511,2839.056,2839.056,<NA>,11.8294,...,14,1,26,Poor,Good,Good,Excellent,Fair,Poor,44
18,980217c4d706469d907be131f329cb0b,2026-01-19,32,2472,961,1511,2839.056,2839.056,<NA>,11.8294,...,14,0,21,Fair,Excellent,Excellent,Excellent,Excellent,Good,48
19,7b57bcd8614a4bbcbef6bc188c82f0a4,2026-01-20,32,1866,355,1511,1892.704,2839.056,<NA>,7.886267,...,13,0,22,Fair,Excellent,Fair,Excellent,Excellent,Excellent,55
20,ad9c64b6ad374864adb28ecc06683bf2,2026-01-21,32,1701,190,1511,1656.116,2839.056,<NA>,6.900483,...,15,1,23,Fair,Good,Excellent,Excellent,Good,Fair,43


### Convert Date columns to `datetime`

In [20]:
df_health1['Date'] = pd.to_datetime(df_health1['Date'])
for col in ['Sleep Start Timestamp GMT', 'Sleep End Timestamp GMT', 'Sleep Start Timestamp Local', 'Sleep End Timestamp Local']:
    df_health1[col] = pd.to_datetime(df_health1[col])

### Export the data

In [22]:
log.info(f'Exporting health data to {path_health_data}')
df_health1.to_csv(path_health_data)

2026-01-22 18:44:28,145 - INFO - Exporting health data to /content/drive/MyDrive/Colab Notebooks/christophermagno/Projects/Personal Health Analysis/personal_health_data_2026.csv


## 🏋🏽‍♂️ Create the Activities Dataframe

In [23]:
remap_activities_columns = {
    'activityId': 'id',
    'activityName': 'Activity Name',
    'activityType': 'Activity Type',

    'startTimeLocal': 'Start Time',
    'endTimeLocal': 'End Time',

    'pr': 'Personal Record',

    'duration': 'Duration',

    'calories': 'Active Calories',
    'bmrCalories': 'Resting Calories',
    'totalCalories': 'Total Calories',

    'steps': 'Steps',
    'distance': 'Distance',

    'averageSpeed': 'Average Speed',
    'maxSpeed': 'Max Speed',

    'averageHR': 'Average Heart Rate',
    'maxHR': 'Max Heart Rate',
    'hrTimeInZone_1': 'Heart Rate Zone 1 Duration',
    'hrTimeInZone_2': 'Heart Rate Zone 2 Duration',
    'hrTimeInZone_3': 'Heart Rate Zone 3 Duration',
    'hrTimeInZone_4': 'Heart Rate Zone 4 Duration',
    'hrTimeInZone_5': 'Heart Rate Zone 5 Duration',

    'lapCount': 'Lap Count',
    'totalSets': 'Total Sets',
    'activeSets': 'Active Sets',
    'totalReps': 'Total Reps',

    'aerobicTrainingEffect': 'Aerobic Training Effect',
    'anaerobicTrainingEffect': 'Anaerobic Training Effect',
    'moderateIntensityMinutes': 'Moderate Intensity Minutes',
    'vigorousIntensityMinutes': 'Vigorous Intensity Minutes',
    'activityTrainingLoad': 'Activity Training Load',
}

In [24]:
activities_data = get_activities_data(dates)
df_activities = pd.DataFrame(activities_data).convert_dtypes()
df_activities['startTimeLocal'] = pd.to_datetime(df_activities['startTimeLocal'])

### Get end time from startTimeLocal and duration

In [25]:
df_activities['endTimeLocal'] = df_activities.apply(lambda row: row['startTimeLocal'] + datetime.timedelta(seconds=row['duration']), axis=1)
df_activities['endTimeLocal'] = pd.to_datetime(df_activities['endTimeLocal'])

### Rename and reorder the columns

In [26]:
df_activities1 = df_activities.rename(columns=remap_activities_columns)[remap_activities_columns.values()]

### Fill some null values

In [27]:
df_activities1[['Distance', 'Average Speed', 'Max Speed']] = df_activities1[['Distance', 'Average Speed', 'Max Speed']].fillna(0)

### Export the data

In [28]:
log.info(f'Exporting activities data to {path_activities_data}')
df_activities1.to_csv(path_activities_data)

2026-01-22 18:44:36,909 - INFO - Exporting activities data to /content/drive/MyDrive/Colab Notebooks/christophermagno/Projects/Personal Health Analysis/personal_acitivites_data_2026.csv
